# ARIA - LITE

ARIA Lite is a lightweight GraphRAG-based biomedical research assistant focused on breast cancer AI literature.

The project combines two retrieval paradigms:

1. Semantic Retrieval
   Dense vector embeddings are used to retrieve papers semantically related to a user query.

2. Graph-Based Retrieval
   Biomedical entities such as genes and drugs are extracted from papers and represented as relationships in a lightweight knowledge graph.

By combining these two approaches, the system aims to provide more grounded and explainable retrieval compared to traditional vector-only RAG systems.

The project is intentionally scoped for rapid iteration and learning:
- ~300-500 PubMed papers
- Abstract-only corpus
- Lightweight graph construction
- Citation-grounded responses

Core technologies:
- PubMed / Entrez API
- SciSpacy
- Sentence Transformers
- FAISS
- Python + Google Colab

End Goal:
Build a small but functional biomedical GraphRAG system capable of retrieving relevant breast cancer AI papers and generating grounded answers with PMID citations.

# 2_data_cleaning_and_preprocessing.ipynb

Purpose:
This notebook prepares raw PubMed biomedical abstracts collected in Week 1 into a clean, structured dataset suitable for:

1. Vector embeddings (semantic retrieval via FAISS)
2. Knowledge graph construction (entity extraction in Week 3)
3. LLM-based retrieval augmentation (GraphRAG-style pipeline)

We perform:
- Deduplication
- Structural normalization of abstracts
- Text cleaning (HTML removal, whitespace normalization)
- Schema standardization

The output of this notebook will be:
- clean_papers.json


In [ ]:
# ============================================================
# SECTION 1 — Imports
# ============================================================

from google.colab import drive
import os

import json
import re
import hashlib
from typing import Dict, List, Any

In [ ]:
# ============================================================
# SECTION 2 — Mount Google Drive
# ============================================================

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# SECTION 3 — Project Paths and data loading
# ============================================================

PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2"

INPUT_PATH = os.path.join(PROJECT_ROOT, "data", "raw", "papers_raw.json")

OUTPUT_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "clean_papers.json")

print("Project root:", PROJECT_ROOT)
print("Input path:", INPUT_PATH)
print("Output path:", OUTPUT_PATH)

with open(INPUT_PATH, "r") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} papers")

Project root: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite
Input path: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite/data/raw/papers_raw.json
Output path: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite/data/processed/clean_papers.json
Loaded 500 papers


In [ ]:
# ============================================================
# SECTION 4 — Utility functions
# ============================================================

def hash_text(text: str) -> str:
    """Create hash for deduplication"""
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def clean_html(text: str) -> str:
    """Remove HTML tags like <sup>+</sup>"""
    return re.sub(r"<.*?>", "", text)


def normalize_whitespace(text: str) -> str:
    """Fix extra spaces, newlines"""
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def flatten_abstract(abstract: Any) -> str:
    """
    Convert structured abstract into single clean text.
    Preserves section structure for better retrieval.
    """

    if isinstance(abstract, str):
        return abstract

    if isinstance(abstract, dict):
        return " ".join([
            f"{section}: {content}"
            for section, content in abstract.items()
        ])

    return ""

In [ ]:
# ============================================================
# SECTION 5 — Core cleaning function
# ============================================================

def clean_record(record: Dict) -> Dict:
    pmid = record.get("pmid")
    title = record.get("title", "")
    abstract = record.get("abstract", "")
    year = record.get("year", None)

    # 1. Flatten abstract
    abstract_text = flatten_abstract(abstract)

    # 2. Clean HTML
    abstract_text = clean_html(abstract_text)

    # 3. Normalize whitespace
    title = normalize_whitespace(title)
    abstract_text = normalize_whitespace(abstract_text)

    # 4. Construct final text
    text = f"Title: {title}. Abstract: {abstract_text}"

    # 5. Year normalization
    try:
        year = int(year) if year else None
    except:
        year = None

    return {
        "pmid": pmid,
        "title": title,
        "abstract_raw": abstract,
        "text": text,
        "year": year,
        "journal": record.get("journal"),
        "authors": record.get("authors", [])
    }

In [ ]:
# ============================================================
# SECTION 6 — Deduplication
# ============================================================

seen_pmids = set()
seen_hashes = set()

cleaned_data = []

for record in raw_data:

    pmid = record.get("pmid")
    if not pmid:
        continue

    # Skip duplicate PMIDs
    if pmid in seen_pmids:
        continue

    cleaned = clean_record(record)

    # Hash-based dedup
    text_hash = hash_text(cleaned["text"])
    if text_hash in seen_hashes:
        continue

    seen_pmids.add(pmid)
    seen_hashes.add(text_hash)

    cleaned_data.append(cleaned)

print(f"Cleaned dataset size: {len(cleaned_data)}")

Cleaned dataset size: 500


In [ ]:
# ============================================================
# SECTION 7 — Filtering small abstracts
# ============================================================

filtered_data = []

for record in cleaned_data:
    if len(record["text"]) < 200:  # threshold can be tuned
        continue
    filtered_data.append(record)

print(f"After filtering: {len(filtered_data)}")

After filtering: 475


In [ ]:
# ============================================================
# SECTION 8 — Save cleaned JSON
# ============================================================

with open(OUTPUT_PATH, "w") as f:
    json.dump(filtered_data, f, indent=2)

print(f"Saved cleaned data to {OUTPUT_PATH}")

Saved cleaned data to /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite/data/processed/clean_papers.json
